# FastAPI와 Slack 웹훅 (RAG Agent 봇)
* 목표: Slack 멘션을 받아 **LangGraph Agent**가 학칙 문서를 검색해 답변

```text
Slack에서 @봇 석사 수업연한이 어떻게 돼?
    → FastAPI /slack/events
    → Agent가 Chroma 검색 + LLM 답변 생성
    → Slack 스레드에 답장
```

오늘 할 일:
1. RAG Agent FastAPI 앱(`slack_app.py`) 만들기
2. 앱 + ngrok 실행하고 Event Subscription에 URL 등록하기
3. Slack에서 멘션해서 Agent 답변이 오는지 확인하기



---
* `day17` 커널
* `.env`에 이미 있어야 할 것:

```text
OPENAI_API_KEY=sk-...
SLACK_BOT_TOKEN=xoxb-...
SLACK_SIGNING_SECRET=...
```

* `16일차_실습/chroma_regulations` — 16.4 노트북에서 만든 Chroma DB
* Slack App에 봇 스코프 `chat:write`, `app_mentions:read` 가 있고, 워크스페이스에 Install 되어 있다고 가정합니다. (16.3에서 만든 앱 재사용 가능)



In [4]:
!pip install fastapi uvicorn slack_sdk python-dotenv langchain-openai langchain-chroma langgraph -q

In [3]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
load_dotenv(Path.cwd().parent / '.env')

WORKDIR = Path.cwd()
CHROMA_DIR = WORKDIR.parent / '16일차_실습' / 'chroma_regulations'
print('WORKDIR :', WORKDIR)
print('CHROMA  :', CHROMA_DIR, '| exists:', CHROMA_DIR.exists())
print('OPENAI_API_KEY set:', bool(os.getenv('OPENAI_API_KEY')))
print('BOT_TOKEN set     :', bool(os.getenv('SLACK_BOT_TOKEN')))
print('SIGNING_SECRET set:', bool(os.getenv('SLACK_SIGNING_SECRET')))



WORKDIR : c:\Users\Admin\OneDrive\바탕 화면\실습\17일차_실습
BOT_TOKEN set     : True
SIGNING_SECRET set: True


---
## 1. RAG Agent FastAPI 앱 만들기
* POST로 온 Slack 이벤트에서 질문을 읽고
* LangGraph ReAct Agent가 `search_regulations` 도구로 Chroma 검색 후 답변 생성
* `chat.postMessage`로 같은 채널 **스레드**에 답장
* Slack 3초 제한 때문에 Agent 호출은 `BackgroundTasks`로 분리

아래 셀을 실행해 `slack_app.py`를 저장하세요.



In [5]:
# RAG Agent 봇 — slack_app.py 생성

slack_app_py = '''
import hashlib
import hmac
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from fastapi import BackgroundTasks, FastAPI, Header, HTTPException, Request
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.prebuilt import create_react_agent
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError

load_dotenv()
load_dotenv(Path(__file__).resolve().parent.parent / ".env")

SLACK_BOT_TOKEN = os.environ["SLACK_BOT_TOKEN"]
SLACK_SIGNING_SECRET = os.environ["SLACK_SIGNING_SECRET"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

CHROMA_DIR = Path(__file__).resolve().parent.parent / "16일차_실습" / "chroma_regulations"
if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        f"{CHROMA_DIR} 없음. 16.4 노트북에서 Chroma 인덱싱을 먼저 완료하세요."
    )

_embedding = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)
_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
_vectorstore = Chroma(persist_directory=str(CHROMA_DIR), embedding_function=_embedding)


@tool
def search_regulations(query: str) -> str:
    """학부·대학원 학칙 문서에서 관련 조항을 검색합니다."""
    docs = _vectorstore.similarity_search(query, k=3)
    if not docs:
        return "관련 문서를 찾지 못했습니다."
    return "\\n\\n".join(
        f"[{d.metadata.get('level', '?')}] {d.page_content[:600]}" for d in docs
    )


_agent = create_react_agent(
    _llm,
    [search_regulations],
    prompt=(
        "당신은 학부·대학원 학칙 안내 봇입니다. "
        "search_regulations로 문서를 검색한 뒤, 검색 결과만 근거로 한국어로 간결히 답하세요. "
        "도구는 최대 2번만 호출하세요."
    ),
)


def run_agent(question: str) -> str:
    result = _agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"recursion_limit": 10},
    )
    answer = result["messages"][-1].content
    if answer.startswith("Sorry, need more steps"):
        return _rag_answer(question)
    return answer


def _rag_answer(question: str) -> str:
    """ponytail: ReAct가 실패하면 17.1식 단순 RAG로 폴백"""
    docs = _vectorstore.similarity_search(question, k=3)
    if not docs:
        return "관련 학칙을 찾지 못했습니다."
    context = "\\n\\n".join(d.page_content[:600] for d in docs)
    prompt = (
        "아래 학칙 발췌만 근거로 질문에 두세 문장으로 답하세요.\\n"
        "근거에 없으면 추측하지 마세요.\\n\\n"
        f"문서:\\n{context}\\n\\n질문: {question}"
    )
    return _llm.invoke(prompt).content


client = WebClient(token=SLACK_BOT_TOKEN)
app = FastAPI(title="Slack RAG Agent Bot")
_seen: set[str] = set()


def verify_slack_signature(body: bytes, timestamp: str | None, signature: str | None) -> None:
    if not timestamp or not signature:
        raise HTTPException(status_code=401, detail="missing signature headers")
    if abs(time.time() - int(timestamp)) > 60 * 5:
        raise HTTPException(status_code=401, detail="stale request")
    base = f"v0:{timestamp}:{body.decode('utf-8')}"
    digest = hmac.new(
        SLACK_SIGNING_SECRET.encode(),
        base.encode(),
        hashlib.sha256,
    ).hexdigest()
    if not hmac.compare_digest(f"v0={digest}", signature):
        raise HTTPException(status_code=401, detail="invalid signature")


def reply_with_agent(channel: str, text: str, thread_ts: str | None = None) -> None:
    print("받은 메시지:", text)
    try:
        answer = run_agent(text)
    except Exception as e:
        answer = f"답변 생성 중 오류가 발생했습니다: {e}"
        print("agent 실패:", e)
    try:
        kwargs: dict = {"channel": channel, "text": answer}
        if thread_ts:
            kwargs["thread_ts"] = thread_ts
        client.chat_postMessage(**kwargs)
    except SlackApiError as e:
        print("postMessage 실패:", e.response.get("error"))


@app.get("/health")
def health():
    return {"ok": True}


@app.post("/slack/events")
async def slack_events(
    request: Request,
    background_tasks: BackgroundTasks,
    x_slack_signature: str | None = Header(default=None),
    x_slack_request_timestamp: str | None = Header(default=None),
):
    body = await request.body()
    verify_slack_signature(body, x_slack_request_timestamp, x_slack_signature)
    payload = await request.json()

    if payload.get("type") == "url_verification":
        return {"challenge": payload.get("challenge")}

    if payload.get("type") != "event_callback":
        return {"ok": True}

    event_id = payload.get("event_id")
    if event_id:
        if event_id in _seen:
            return {"ok": True}
        _seen.add(event_id)

    event = payload.get("event") or {}
    if event.get("bot_id") or event.get("subtype") == "bot_message":
        return {"ok": True}

    text = (event.get("text") or "").strip()
    channel = event.get("channel")
    if not text or not channel:
        return {"ok": True}

    if event.get("type") == "app_mention":
        parts = text.split(maxsplit=1)
        text = parts[1] if len(parts) > 1 else text

    if not text:
        return {"ok": True}

    # ponytail: Slack은 3초 안에 200 응답 필요 — Agent는 백그라운드에서 실행
    background_tasks.add_task(reply_with_agent, channel, text, event.get("ts"))
    return {"ok": True}
'''

path = WORKDIR / 'slack_app.py'
path.write_text(slack_app_py.strip() + '\n', encoding='utf-8')
print('저장:', path)



저장: c:\Users\Admin\OneDrive\바탕 화면\실습\17일차_실습\slack_app.py


앱이 하는 일 요약:

| 요청 | 동작 |
|------|------|
| `url_verification` | `challenge` 그대로 반환 (URL 등록용) |
| `event_callback` + `app_mention` | Agent가 Chroma 검색 후 답변을 스레드에 전송 |
| 봇 자신의 메시지 | 무시 (무한 루프 방지) |
| Agent 호출 | `BackgroundTasks` — Slack 3초 제한 회피 |



---
## 2. 앱 실행 → ngrok → Event Subscription (자세히)

**서버가 켜진 상태**에서 Slack에 URL을 넣어야 Verified가 됩니다.

### Step A. FastAPI 서버 켜기 (터미널 1)

```bash
conda activate day17
cd 17일차_실습
uvicorn slack_app:app --reload --port 8000
```

성공하면 대략 이렇게 보입니다.

```text
Uvicorn running on http://127.0.0.1:8000
```

브라우저에서 http://127.0.0.1:8000/health 를 열면 `{"ok":true}` 가 보이면 좋습니다.

> 이 터미널은 끄지 말고 그대로 둡니다.

---

### Step B. ngrok 설치 (처음 한 번)

로컬 `8000` 포트는 인터넷에서 안 보이므로, Slack이 우리 PC로 POST를 보내게 **공개 HTTPS 주소**가 필요합니다.

1. https://ngrok.com/download 에서 Windows용 설치 (또는 `winget install ngrok`)
2. ngrok 가입 후 대시보드의 authtoken 등록:

```bash
ngrok config add-authtoken <본인_토큰>
```

---

### Step C. ngrok으로 8000 포트 열기 (터미널 2)

**새 터미널**을 열고:

```bash
ngrok http 8000
```

화면에 비슷한 줄이 나옵니다.

```text
Forwarding    https://abcd-12-34-56.ngrok-free.app -> http://127.0.0.1:8000
```

* 여기서 **`https://....ngrok-free.app`** 부분만 복사합니다. (`http` 말고 `https`)
* ngrok을 끄거나 재시작하면 주소가 **바뀝니다**. 바뀌면 Slack URL도 다시 넣어야 합니다.

> 터미널 1(uvicorn)과 터미널 2(ngrok) **둘 다 켜 둔 상태**로 다음으로 갑니다.

---

### Step D. Slack Event Subscriptions에 URL 등록

1. 브라우저에서 https://api.slack.com/apps 접속
2. 사용할 **App** 클릭
3. 왼쪽 메뉴 **Event Subscriptions**
4. **Enable Events** 를 On
5. **Request URL** 칸에 아래처럼 입력 (본인 ngrok 주소로 바꿔서)

```text
https://abcd-12-34-56.ngrok-free.app/slack/events
```

주의:
* 끝 경로가 반드시 `/slack/events`
* `uvicorn` + `ngrok` 이 둘 다 살아 있어야 함

6. 잠시 기다리면 URL 옆에 **Verified** ✔ 가 뜹니다.

Verified가 안 되면 흔한 원인:
* uvicorn이 안 켜져 있음
* ngrok이 다른 포트를 보고 있음 (`8000`인지 확인)
* URL 오타 / `/slack/events` 빠짐
* Signing Secret이 `.env`와 Slack App의 값이 다름

---

### Step E. 구독할 이벤트 선택

같은 Event Subscriptions 페이지 아래 **Subscribe to bot events** 에서:

* `app_mention` 추가 (채널에서 `@봇` 했을 때)

필요하면:
* `message.im` (DM)

맨 아래 **Save Changes** 클릭.

워크스페이스에 앱을 다시 설치하라는 안내가 뜨면 **reinstall** 합니다.

---

### Step F. 채널에 봇 초대

테스트할 Slack 채널에서:

```text
/invite @봇이름
```



---
## 3. Slack에서 확인하기

채널에 입력:

```text
@봇이름 석사 수업연한이 어떻게 돼?
```

기대한 결과:
* 멘션한 메시지 **스레드**에 학칙 근거 답변이 돌아옴
* uvicorn 터미널에 `받은 메시지: ...` 로그가 찍힘

안 되면 체크:
1. Event Subscriptions가 Verified 인가?
2. `app_mention` 구독 + Save 했는가?
3. `/invite`로 봇을 채널에 넣었는가?
4. ngrok URL이 바뀌지 않았는가?
5. 봇 토큰에 `chat:write` 권한이 있는가?
6. `16일차_실습/chroma_regulations` 폴더가 있는가? (16.4 선행)
7. `.env`에 `OPENAI_API_KEY`가 있는가?



---
## 정리
1. `slack_app.py` — LangGraph ReAct Agent + Chroma RAG로 학칙 질문에 답함
2. `uvicorn` + `ngrok` → Slack Request URL Verified
3. `@봇` 멘션으로 Agent 왕복 확인

Agent 구조:
* `search_regulations` 도구 → 17.1과 같은 Chroma DB 검색
* `create_react_agent` → 필요 시 도구를 여러 번 호출하며 답변 생성
* `BackgroundTasks` → Slack 3초 응답 제한을 지키면서 LLM 실행

